# 03. Demo-коннектор (issue tracker)

Первая внешняя capability: локальный issue-tracker коннектор.
Он не требует ключей, поэтому удобен для live-демонстрации.

Генерируется `agents/step_03_demo.py`. Запуск:

```bash
uv run langgraph dev --config langgraph.steps.json
```

## Визуальная рамка

![Connector payload flow](visuals/03_connector_payload.svg)

Этот шаг показывает главный connector pattern: внешний источник превращается в структурированный payload, который затем становится контекстом для LLM.


In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend, LocalShellBackend
from dotenv import load_dotenv

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR if (NOTEBOOK_DIR / 'pyproject.toml').exists() else NOTEBOOK_DIR.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

load_dotenv(REPO_ROOT / '.env')

from connectors.demo import DEMO_TOOLS, list_demo_issues, get_demo_issue

# Пробуем tools напрямую вне агента
print(list_demo_issues.invoke({}))
print(get_demo_issue.invoke({"issue_id": "DEMO-101"}))

Теперь соберём агента с этими tools.

In [ ]:
DEFAULT_MODEL = 'openrouter:tencent/hy3:free'

def model_name() -> str:
    return os.getenv('OPENCLAW_MODEL', DEFAULT_MODEL)


def workspace_root() -> Path:
    return Path(os.getenv('OPENCLAW_WORKSPACE', '.')).expanduser().resolve()


def backend():
    root_dir = workspace_root()
    if os.getenv('OPENCLAW_ENABLE_LOCAL_SHELL') != '1':
        return FilesystemBackend(root_dir=root_dir, virtual_mode=True)
    return LocalShellBackend(
        root_dir=root_dir, virtual_mode=True, inherit_env=True,
        timeout=120, max_output_bytes=80_000,
    )


BASE_PROMPT = """\
You are OpenClaw, an experimental open-source coding agent built with LangChain
Deep Agents. You help with software engineering, repository navigation, product
research, and implementation.

Work like a careful senior engineer:
- inspect before editing;
- keep changes focused;
- verify with tests or equivalent checks;
- explain only the useful result to the user.
"""


agent = create_deep_agent(
    model=model_name(),
    tools=[*DEMO_TOOLS],
    system_prompt=BASE_PROMPT,
    backend=backend(),
    interrupt_on={"execute": True},
)

print(type(agent).__name__)

### Сгенерировать entrypoint

In [ ]:
import json

AGENTS_DIR = REPO_ROOT / "agents"
AGENTS_DIR.mkdir(exist_ok=True)

agent_code = '''"""Step 03: adds demo issue-tracker connector tools."""

from __future__ import annotations

import os
from pathlib import Path

from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend, LocalShellBackend
from dotenv import load_dotenv

from connectors.demo import DEMO_TOOLS

load_dotenv()

DEFAULT_MODEL = "openrouter:tencent/hy3:free"


def _workspace_root() -> Path:
    return Path(os.getenv("OPENCLAW_WORKSPACE", ".")).expanduser().resolve()


def _backend():
    root_dir = _workspace_root()
    if os.getenv("OPENCLAW_ENABLE_LOCAL_SHELL") != "1":
        return FilesystemBackend(root_dir=root_dir, virtual_mode=True)
    return LocalShellBackend(
        root_dir=root_dir, virtual_mode=True, inherit_env=True,
        timeout=120, max_output_bytes=80_000,
    )


agent = create_deep_agent(
    model=os.getenv("OPENCLAW_MODEL", DEFAULT_MODEL),
    tools=[*DEMO_TOOLS],
    system_prompt="""\
You are OpenClaw, an experimental open-source coding agent built with LangChain
Deep Agents. You help with software engineering, repository navigation, product
research, and implementation.

Work like a careful senior engineer:
- inspect before editing;
- keep changes focused;
- verify with tests or equivalent checks;
- explain only the useful result to the user.
""",
    backend=_backend(),
    interrupt_on={"execute": True},
)
'''

step_file = AGENTS_DIR / "step_03_demo.py"
step_file.write_text(agent_code)

steps_config = {
    "dependencies": ["."],
    "graphs": {
        "openclaw": "./agents/step_03_demo.py:agent",
    },
    "env": ".env",
}
config_path = REPO_ROOT / "langgraph.steps.json"
config_path.write_text(json.dumps(steps_config, indent=2) + "\n")

print(f"Wrote {step_file}")
print(f"Wrote {config_path}")
print()
print("Restart:")
print("  uv run langgraph dev --config langgraph.steps.json")

### Проверочный prompt

После перезапуска попробуй:

> "Use the demo issue connector. List issues, fetch DEMO-101, and propose the next implementation step."